# Stock Price Prediction using LSTM
This notebook demonstrates how to fetch historical stock data, preprocess it, build a Long Short-Term Memory (LSTM) neural network using Keras, and save the trained model for the web application.

In [ ]:
import pandas as pd
import numpy as np
from yahooquery import Ticker
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense, LSTM
import matplotlib.pyplot as plt

## 1. Fetching Data
We use `yahooquery` to fetch the historical data for our target stock.

In [ ]:
ticker_symbol = 'AAPL'
t = Ticker(ticker_symbol)
df = t.history(start='2012-01-01', end='2024-01-01')
df = df.reset_index()
df.set_index('date', inplace=True)
df.index = pd.to_datetime(df.index)
df.sort_index(inplace=True)
df.head()

In [ ]:
# Visualize the closing price history
plt.figure(figsize=(16,8))
plt.title('Close Price History')
plt.plot(df['close'])
plt.xlabel('Date', fontsize=18)
plt.ylabel('Close Price USD ($)', fontsize=18)
plt.show()

## 2. Preprocessing
Scale the data and create the training datasets with a 100-day lookback.

In [ ]:
data = df.filter(['close'])
dataset = data.values
training_data_len = int(np.ceil( len(dataset) * .7 ))

scaler = MinMaxScaler(feature_range=(0,1))
scaled_data = scaler.fit_transform(dataset)

train_data = scaled_data[0:int(training_data_len), :]
x_train = []
y_train = []

for i in range(100, len(train_data)):
    x_train.append(train_data[i-100:i, 0])
    y_train.append(train_data[i, 0])
    
x_train, y_train = np.array(x_train), np.array(y_train)
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))

## 3. Building the LSTM Model

In [ ]:
model = Sequential()
model.add(LSTM(50, return_sequences=True, input_shape=(x_train.shape[1], 1)))
model.add(LSTM(50, return_sequences=False))
model.add(Dense(25))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

## 4. Training and Saving the Model

In [ ]:
model.fit(x_train, y_train, batch_size=32, epochs=10)
model.save('keras_model.h5')
print('Model successfully saved as keras_model.h5')